# ZIP-RC: Zero-Overhead Introspection for Adaptive Test-Time Compute

This notebook runs the complete ZIP-RC pipeline from start to finish. ZIP-RC enables LLMs to predict their own success probability (reward) and computational cost (remaining generation length) during inference—without adding any extra overhead.

**Pipeline Overview:**
1. **Setup** - Install dependencies and authenticate with HuggingFace
2. **Generate Rollouts** - Generate completions from the base model
3. **Build Dataset** - Create prefix training examples with joint labels
4. **Train ZIP-RC** - Fine-tune the model in single-model mode
5. **Inspect Checkpoints** - Read the metrics saved by the training script
6. **Test Inference** - Verify the trained model can introspect

---

## 1. Setup & Environment Check

First, let's install the required dependencies and check our GPU availability.

In [ ]:
# Install required packages
!pip install -q --upgrade 'transformers>=4.51.1' accelerate torch datasets huggingface_hub bitsandbytes
print("Packages installed!")

In [ ]:
# Check GPU availability and memory
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU available: {gpu_name}")
    print(f"   Total memory: {gpu_memory:.1f} GB")
    
    # Memory warning for the shared Qwen 8B model
    if gpu_memory < 16:
        print("\nWARNING: Qwen/Qwen3-8B usually needs about 16GB VRAM in float16.")
        print("   You may encounter OOM errors. Consider:")
        print("   - Using Colab Pro for A100/V100 GPU")
        print("   - Using a smaller model")
        print("   - Reducing batch size or sequence length")
    else:
        print("\nGPU memory looks sufficient for Qwen/Qwen3-8B")
else:
    print("No GPU available!")
    print("   Go to Runtime -> Change runtime type -> GPU")
    print("   This notebook requires a GPU to run.")

In [ ]:
# HuggingFace authentication (required for model downloads)
from huggingface_hub import notebook_login

print("Login to HuggingFace for model downloads")
print("   Using model: Qwen/Qwen3-8B")
print("   Then enter your HF token below:\n")

notebook_login()

## 2. Get the Code

Clone the ZIP-RC repository and run the checked-in scripts directly. This keeps Colab on the exact same code path as the standalone scripts.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/rshiramss/zip-rc"
REPO_DIR = Path("/content/zip-rc")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Using repo at {REPO_DIR}")
print("The notebook will run the repo scripts directly.")

### Verify Repo Files

This notebook uses the checked-in script files as the source of truth so Colab matches the repo logic exactly.

In [ ]:
from pathlib import Path
from zip_rc_model import DEFAULT_MODEL_NAME

required_files = [
    Path("zip_rc_model.py"),
    Path("generate_rollouts.py"),
    Path("build_prefix_dataset.py"),
    Path("train_zip_rc.py"),
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing repo files: {missing}")

print("Repo files found:")
for path in required_files:
    print(f"  - {path}")
print(f"\nUsing shared default model: {DEFAULT_MODEL_NAME}")

## 3. Create Sample Data

Let's create a small set of math problems for our pipeline demo.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

NUM_EXAMPLES = 10
NUM_SAMPLES = 1
MAX_NEW_TOKENS = 64
LOAD_IN_8BIT = False

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"num_examples={NUM_EXAMPLES}, num_samples={NUM_SAMPLES}, max_new_tokens={MAX_NEW_TOKENS}")
print(f"load_in_8bit={LOAD_IN_8BIT}")

## 4. Step 1: Generate Rollouts

Generate completions with the repo's `generate_rollouts.py` script.

**What this does:**
- Uses the exact script logic from the repo
- Pulls GSM8K examples the same way as local runs
- Saves rollouts with model metadata for later steps

In [ ]:
import json
import subprocess

ROLLOUT_PATH = "data/rollouts.jsonl"
command = [
    "python",
    "generate_rollouts.py",
    "--num-examples", str(NUM_EXAMPLES),
    "--num-samples", str(NUM_SAMPLES),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--greedy",
    "--output", ROLLOUT_PATH,
]
if LOAD_IN_8BIT:
    command.append("--load-in-8bit")

print("Running:", " ".join(command))
subprocess.run(command, check=True)

with open(ROLLOUT_PATH) as f:
    rollouts = [json.loads(line) for line in f]

print(f"\nGenerated {len(rollouts)} rollouts")
for row in rollouts[:3]:
    print(f"  Q: {row['question']}")
    print(f"  Reward: {row['reward']} | Predicted: {row['predicted_answer']} | Truth: {row['ground_truth']}")
    print()

In [ ]:
# Free up any cached GPU memory before the next step
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleared GPU cache")

## 5. Step 2: Build Prefix Dataset

Build the prefix dataset with the repo's `build_prefix_dataset.py` script.

**What this does:**
- Reads the rollout file from the previous script step
- Resolves the tokenizer model from rollout metadata
- Writes one prefix example per completion prefix

In [ ]:
import json
import subprocess
from collections import Counter

PREFIX_PATH = "data/prefix_dataset.jsonl"
command = [
    "python",
    "build_prefix_dataset.py",
    "--input", ROLLOUT_PATH,
    "--output", PREFIX_PATH,
]

print("Running:", " ".join(command))
subprocess.run(command, check=True)

with open(PREFIX_PATH) as f:
    examples = [json.loads(line) for line in f]

print(f"\nBuilt {len(examples)} prefix examples")
for label, count in sorted(Counter(ex["joint_label"] for ex in examples).items())[:10]:
    print(f"  label {label}: {count}")

## 6. Step 3: Train ZIP-RC Model

Train with the repo's `train_zip_rc.py` script.

**What this does:**
- Uses the same argument parsing and validation as local script runs
- Enforces a single-model training setup
- Saves checkpoints to the repo `checkpoints/` directory

In [ ]:
import subprocess

CHECKPOINT_DIR = "checkpoints"
TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 2
TRAIN_LR = 1e-4

command = [
    "python",
    "train_zip_rc.py",
    "--data_path", PREFIX_PATH,
    "--epochs", str(TRAIN_EPOCHS),
    "--batch_size", str(TRAIN_BATCH_SIZE),
    "--learning_rate", str(TRAIN_LR),
    "--checkpoint_dir", CHECKPOINT_DIR,
    "--alpha_kl", "0.0",
]

print("Running:", " ".join(command))
subprocess.run(command, check=True)

## 7. Step 4: Inspect Checkpoints

Read back the checkpoint files written by `train_zip_rc.py`.

In [ ]:
OUTPUT_DIR = CHECKPOINT_DIR
print(f"Using checkpoint directory: {OUTPUT_DIR}")

In [ ]:
from pathlib import Path
import torch

checkpoint_dir = Path(CHECKPOINT_DIR)
checkpoints = sorted(checkpoint_dir.glob("checkpoint_*.pt"))
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")

latest_checkpoint = checkpoints[-1]
checkpoint = torch.load(latest_checkpoint, map_location="cpu")
print(f"Latest checkpoint: {latest_checkpoint}")
print(f"Metrics: {checkpoint['metrics']}")

In [ ]:
# The latest checkpoint is already loaded above as `checkpoint`.
print(f"Ready to use checkpoint from: {latest_checkpoint}")

## 8. Step 5: Test Inference

Let's test our trained ZIP-RC model! We'll see if it can predict:
- **Expected Reward**: Probability of getting the right answer (0-1)
- **Expected Length**: Which length bin the remaining tokens fall into (0-4)

In [ ]:
import torch
from zip_rc_model import DEFAULT_MODEL_NAME, ZipRCModel, LENGTH_BIN_EDGES

# Length bin descriptions
LENGTH_BIN_NAMES = ["0-9 tokens", "10-19 tokens", "20-39 tokens", "40-79 tokens", "80+ tokens"]

def test_introspection(model, tokenizer, prompt: str, partial_answer: str = ""):
    """Test ZIP-RC introspection on a prompt + partial answer."""
    text = f"Q: {prompt}\nA:{partial_answer}"
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        expected_reward, expected_length = model.predict_zip(**inputs)

    reward = expected_reward.item()
    length_bin = int(round(expected_length.item()))
    length_bin = max(0, min(length_bin, 4))

    print(f"Input: '{text}'")
    print(f"  Expected Reward: {reward:.3f} ({'likely correct' if reward > 0.5 else 'likely wrong'})")
    print(f"  Expected Length Bin: {length_bin} ({LENGTH_BIN_NAMES[length_bin]})")
    print()

    return reward, length_bin

model = ZipRCModel(DEFAULT_MODEL_NAME, freeze_backbone=True)
model.load_state_dict(checkpoint["model_state_dict"], strict=False)
model.eval()

tokenizer = model.tokenizer

print("Testing ZIP-RC Introspection\n")
print("=" * 60)

for prompt, partial in [
    ("What is 15 + 27?", ""),
    ("What is 15 + 27?", " 4"),
    ("What is 15 + 27?", " 42"),
    ("What is 8 * 7?", ""),
    ("What is 8 * 7?", " 56"),
]:
    test_introspection(model, tokenizer, prompt, partial)

## 9. Save Checkpoint to Google Drive (Optional)

Download your trained model to Google Drive for later use.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy checkpoints to Drive
import shutil

DRIVE_PATH = "/content/drive/MyDrive/zip_rc_checkpoints"
shutil.copytree(OUTPUT_DIR, DRIVE_PATH, dirs_exist_ok=True)

print(f"Checkpoints saved to Google Drive: {DRIVE_PATH}")

## Summary

You have now run the notebook against the same repo scripts used outside Colab:

1. **Setup** - Installed dependencies and authenticated with HuggingFace
2. **Generate Rollouts** - Ran `generate_rollouts.py`
3. **Build Dataset** - Ran `build_prefix_dataset.py`
4. **Train ZIP-RC** - Ran `train_zip_rc.py` in single-model mode
5. **Test Inference** - Reloaded the latest checkpoint for a quick introspection check

### Key Insights

- **Zero Overhead**: The ZIP tokens are computed in the same forward pass as text tokens
- **Joint Prediction**: The model predicts both reward (will I succeed?) and length (how many tokens left?)
- **Single-Model Pipeline**: The notebook runs the same checked-in scripts and shared default model as local runs

### Next Steps

- Train on more diverse prompts and longer completions
- Use the trained model for Best-of-N selection
- Implement beam search with ZIP-guided pruning
- Experiment with different binning strategies